In [1]:
#Assignment 7: Gabriel Maldonado

In [2]:
import pygame
import moderngl
import numpy
import math
import glm
import assimp_py
from loadModelUsingAssimp_V1 import getObjectDataList

pygame.init()

width = 840
height = 480
pygame.display.gl_set_attribute(pygame.GL_CONTEXT_PROFILE_MASK, pygame.GL_CONTEXT_PROFILE_CORE)
display = pygame.display.set_mode((width, height), flags= pygame.OPENGL | pygame.DOUBLEBUF | pygame.RESIZABLE)
display = pygame.display.set_caption(title = "Assignment 7 - Gabriel Maldonado")
gl = moderngl.get_context()
gl.info["GL_VERSION"]


texture_vertex_shader_code = '''
    #version 330 core

    layout(location=0) in vec3 positions;
    layout (location=1) in vec2 inUV;
    out vec2 uv;
    uniform mat4 model;
    uniform mat4 viewMatrix;
    uniform mat4 perspectiveMatrix;

    void main(){
        gl_Position = perspectiveMatrix * viewMatrix * model * vec4(positions,1);
        uv = inUV;
}

'''


texture_fragment_shader_code = '''
    #version 330 core

    in vec2 uv;
    uniform sampler2D tex;
    out vec4 tColor;

    void main(){
        tColor = texture(tex, uv);
}    
'''


textureProgram = gl.program(
    vertex_shader = texture_vertex_shader_code,
    fragment_shader = texture_fragment_shader_code
)

geomDataList, indexDataList, bounds, texNames, scene = getObjectDataList("./rayman/scene.gltf")
vars(bounds)


C:\Users\vanes\AppData\Roaming\Python\Python313\site-packages\pygame\pkgdata.py:25: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  from pkg_resources import resource_stream, resource_exists


pygame 2.6.1 (SDL 2.28.4, Python 3.13.7)
Hello from the pygame community. https://www.pygame.org/contribute.html
8  different sets of Texture coordinates. Only the first one is loaded.
8  different sets of Texture coordinates. Only the first one is loaded.
8  different sets of Texture coordinates. Only the first one is loaded.
8  different sets of Texture coordinates. Only the first one is loaded.
8  different sets of Texture coordinates. Only the first one is loaded.


{'boundingBox': [vec3( -0.250586, -0.168069, 0.0677629 ),
  vec3( -0.0779052, 0.0121503, 0.140022 )],
 'center': vec3( -0.164245, -0.0779594, 0.103893 ),
 'radius': 0.12992218136787415}

In [ ]:


renderables = []
for index, geomData in enumerate(geomDataList):
    geomBuffer = gl.buffer(geomData)
    indexBuffer = gl.buffer(indexDataList[index])
    textureRenderable = gl.vertex_array(textureProgram, [(geomBuffer, "3f 2f", "positions", "inUV")], index_buffer = indexBuffer, index_element_size=4)
    renderables.append(textureRenderable)

samplers = []
for i, img in enumerate(texNames):
    texture_img = pygame.image.load(img)
    texture_data = pygame.image.tobytes(texture_img, "RGB", True)
    texture = gl.texture(texture_img.get_size(), 3, data = texture_data)
    texture.build_mipmaps()
    sampler = gl.sampler(texture = texture, repeat_x = True, repeat_y = True, filter = (gl.LINEAR_MIPMAP_LINEAR, gl.LINEAR))
    samplers.append(sampler)
    sampler.use(i)

def recursive_render(node, M):
    if node is None:
        print("Empty node")
        return
    #print("Number of meshes: ", len(node.meshes))
    nodeTransform = glm.transpose(glm.mat4(node.transformation))
    currentTransform = M * nodeTransform
    if node.num_meshes > 0:
        for index in node.mesh_indices:
            textureProgram["tex"] = index
            textureProgram["model"].write(currentTransform)
            renderables[index].render()
            
    for node in node.children:
        recursive_render(node, currentTransform)
        
def render():
    recursive_render(scene.root_node, glm.mat4(1))

#textureRenderable = gl.vertex_array(textureProgram, [
 #   (geomDataList, "3f 2f", "positions", "inUV")]
#)



gl.enable(gl.DEPTH_TEST)
pause = False
running = True
clock = pygame.time.Clock()
camera_elevation_angle = 30
camera_y_angle = 0
camera_angle_in_radians  = math.radians(90 - camera_elevation_angle)
distance = 2 * bounds.radius
displacement_vector = glm.vec3(0, distance * math.cos(camera_angle_in_radians), distance * math.sin(camera_angle_in_radians))


while running:
    for event in pygame.event.get():
        if event.type == pygame.QUIT:
            running = False
        elif (event.type == pygame.KEYDOWN):
            if event.key == 27:
                running = False
            elif event.key == pygame.K_p:
                pause = not pause
        elif (event.type == pygame.WINDOWRESIZED):
            width = event.x
            height = event.y
            #perspectiveMatrix = glm.perspective(fov_radian, width/height, near, far)

    aspect = width/height

    eye_y = distance * math.cos(camera_angle_in_radians)

    eye_radius = distance * math.sin(camera_angle_in_radians)
    eye_x = eye_radius * math.cos(math.radians(camera_angle_in_radians))
    eye_z = eye_radius * math.sin(math.radians(camera_angle_in_radians))

    eye_point = glm.vec3(bounds.center) + glm.rotate(displacement_vector, camera_y_angle, glm.vec3(0,1,0))
    target_point = glm.vec3(bounds.center)
    up_vector = glm.vec3(0, 1, 0)
    viewMatrix = glm.lookAt(eye_point, target_point, up_vector)    

    camera_perspective_matrix = glm.perspective(math.radians(45), aspect, bounds.radius, 3 * bounds.radius)         
            
    gl.clear(0.5,0.5,0)

    textureProgram["viewMatrix"].write(viewMatrix)
    textureProgram["perspectiveMatrix"].write(camera_perspective_matrix)
    textureProgram["model"].write(glm.mat4(1))
    #textureRenderable.render()
    render()

    
    pygame.display.flip()
    if not pause:
        clock.tick(30)
        camera_y_angle = camera_y_angle + .1
        if camera_y_angle > 360:
            camera_y_angle = 0
    
pygame.display.quit()


In [ ]:
renderables

In [ ]:
samplers